In [7]:
import time
import torch
import os
import IPython.display as display

if not hasattr(display, "set_matplotlib_formats"):
    # no-op fallback (or you can force retina/png instead)
    def set_matplotlib_formats(*args, **kwargs):
        return None
    display.set_matplotlib_formats = set_matplotlib_formats

from d2l.torch import Animator

from net import Net
from utils import load_val_dataset, load_test_dataset, build_pyg_data
from faco import MFACO_TSP

from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
PRETRAINED_DIR = PROJECT_ROOT / "../pretrained"
torch.manual_seed(1234)

EPS = 1e-10
H=200
disable_heuristic = False
use_local_search = True
extend_ls = True
smooth_mmas = True
rho = 0.1

device = 'cuda:0'

# Import your compiled extension (needed for get_max_threads print below)
try:
    import faco_tsp
except Exception as e:
    faco_tsp = None
    print("Warning: could not import faco_tsp:", e)

if faco_tsp is not None:
    print("faco_tsp imported. OpenMP max threads:", faco_tsp.get_max_threads())

faco_tsp imported. OpenMP max threads: 10


In [8]:

import numpy as np
import torch

def _popcount_i64(x: torch.Tensor) -> torch.Tensor:
    """
    Vectorized popcount for int64 tensor x (CUDA-safe).
    Interprets bits in two's complement; for non-negative values this matches uint64 popcount.
    Returns int64 counts in [0, 64].
    """
    if x.dtype != torch.int64:
        raise TypeError(f"_popcount_i64 expects torch.int64, got {x.dtype}")

    # Masks as int64 constants (fit within signed range)
    m1  = x.new_tensor(0x5555555555555555, dtype=torch.int64)
    m2  = x.new_tensor(0x3333333333333333, dtype=torch.int64)
    m4  = x.new_tensor(0x0F0F0F0F0F0F0F0F, dtype=torch.int64)
    h01 = x.new_tensor(0x0101010101010101, dtype=torch.int64)

    # Assumes x is non-negative (your valid_mask should be)
    x = x - ((x >> 1) & m1)
    x = (x & m2) + ((x >> 2) & m2)
    x = (x + (x >> 4)) & m4
    return (x * h01) >> 56


def replay_logp_from_cpp_batch_trace(traces, prob_sparse: torch.Tensor):
    device = prob_sparse.device
    k = int(prob_sparse.size(1))
    if k > 64:
        raise ValueError(f"k={k} > 64 but trace.valid_mask is 64-bit; store n_valid explicitly or widen mask.")

    # Trace arrays -> torch (non-diff)
    curr = torch.as_tensor(np.asarray(traces.curr_nodes, dtype=np.int64), device=device)
    is_stoch = torch.as_tensor(np.asarray(traces.is_stochastic, dtype=np.uint8), device=device).bool()
    used_unif = torch.as_tensor(np.asarray(traces.used_uniform, dtype=np.uint8), device=device).bool()
    pick = torch.as_tensor(np.asarray(traces.pick_j, dtype=np.int64), device=device)

    # IMPORTANT: keep mask as int64 on GPU to allow indexing
    vm_i64 = torch.as_tensor(np.asarray(traces.valid_mask, dtype=np.uint64), device=device).to(torch.int64)

    # Ant mapping on GPU
    starts_t = torch.as_tensor(np.asarray(traces.starts, dtype=np.int64), device=device, dtype=torch.int64)
    n_ants = int(getattr(traces, "n_ants", int(starts_t.numel() - 1)))
    counts_t = (starts_t[1:1+n_ants] - starts_t[:n_ants]).to(torch.int64)

    ant_idx_all = torch.repeat_interleave(
        torch.arange(n_ants, device=device, dtype=torch.int64),
        counts_t,
    )

    # ndec per ant (stochastic only)
    if bool(is_stoch.any().item()):
        ndec = torch.bincount(ant_idx_all[is_stoch], minlength=n_ants).to(torch.int32)
    else:
        ndec = torch.zeros((n_ants,), device=device, dtype=torch.int32)

    logp = torch.zeros((n_ants,), device=device, dtype=torch.float32)

    # --- Roulette steps ---
    roulette = is_stoch & (~used_unif) & (pick >= 0)
    if bool(roulette.any().item()):
        idx = roulette.nonzero(as_tuple=False).squeeze(1)
        curr_r = curr[idx]
        pick_r = pick[idx]

        in_range = (pick_r >= 0) & (pick_r < k)
        if not bool(in_range.all().item()):
            bad = pick_r[~in_range][:10].detach().cpu().tolist()
            raise ValueError(f"pick_j out of range (k={k}). Examples: {bad}")

        vm_r = vm_i64[idx]  # int64, CUDA-indexable
        w = prob_sparse[curr_r]  # (D_r, k), differentiable

        # Build validity matrix from bitmask (all int64 ops, then cast)
        bitpos = torch.arange(k, device=device, dtype=torch.int64)
        valid = ((vm_r.unsqueeze(1) >> bitpos) & 1).to(w.dtype)

        denom = (w * valid).sum(dim=1).clamp_min(1e-12)
        numer = w.gather(1, pick_r.unsqueeze(1)).squeeze(1).clamp_min(1e-12)

        chosen_valid = (((vm_r >> pick_r) & 1) != 0)
        if not bool(chosen_valid.all().item()):
            raise ValueError("Trace inconsistency: pick_j not valid under valid_mask for some roulette decisions.")

        lp = torch.log(numer / denom)
        logp.scatter_add_(0, ant_idx_all[idx], lp)

    # --- Uniform steps ---
    unif = is_stoch & used_unif
    if bool(unif.any().item()):
        idx = unif.nonzero(as_tuple=False).squeeze(1)
        vm_u = vm_i64[idx]  # int64
        m = _popcount_i64(vm_u).to(torch.float32)
        lp = -torch.log(m.clamp_min(1.0))
        logp.scatter_add_(0, ant_idx_all[idx], lp)

    return logp, ndec

In [9]:

def infer_instance(model, coords, k_sparse, n_ants, dynamic):
    if model is not None:
        model.eval()

    aco = MFACO_TSP(
        n_ants=n_ants,
        coords=coords,
        cand_list_size=k_sparse,
        backup_list_size=k_sparse,
        disable_heuristic=disable_heuristic,
        use_local_search=use_local_search,
        decay=rho,
        device=device,
        enable_torch_sync=True,
        extend_ls=extend_ls,
        smooth_mmas=smooth_mmas,
    )

    best_seen = float("inf")
    avg_last = None
    heu_mat = None

    for _ in range(H):
        if model is not None:
            pyg_data = build_pyg_data(aco, coords, device, dynamic=dynamic)
            heu_vec = model(pyg_data).view(-1)
            heu_mat = heu_vec.view(aco.n, aco.k) + EPS

        costs, flats, _, _, _ = aco.sample(require_prob=False, prior=heu_mat)

        avg_last = float(costs.mean())
        best_idx = int(costs.argmin())
        best_cost = float(costs[best_idx])
        best_seen = min(best_seen, best_cost)

        aco._update_pheromone_from_flat(flats[best_idx], best_cost)

    return avg_last, best_seen

In [10]:


@torch.no_grad()
def validation(n_ants, net, val_dataset, k_sparse, dynamic):
    sum_sample_best, sum_aco_best = 0, 0
    
    n = len(val_dataset)
    for coords in val_dataset:
        avg_last, best_seen = infer_instance(net, coords, k_sparse, n_ants, dynamic)
        sum_sample_best += avg_last; sum_aco_best += best_seen
    
    n_val = len(val_dataset)
    avg_last = sum_sample_best/n_val
    avg_aco_best = sum_aco_best/n_val
    
    return avg_last, avg_aco_best

In [ ]:
import os, tempfile, subprocess
import numpy as np
import torch

def write_tsplib_euc2d(path, coords_int, name="inst"):
    """
    coords_int: (n,2) int array
    """
    n = coords_int.shape[0]
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"NAME : {name}\n")
        f.write("TYPE : TSP\n")
        f.write(f"DIMENSION : {n}\n")
        f.write("EDGE_WEIGHT_TYPE : EUC_2D\n")
        f.write("NODE_COORD_SECTION\n")
        for i, (x, y) in enumerate(coords_int, start=1):
            f.write(f"{i} {int(x)} {int(y)}\n")
        f.write("EOF\n")

def write_lkh_par(path, tsp_path, out_path, lkh_runs=10, seed=1234):
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"PROBLEM_FILE = {tsp_path}\n")
        f.write(f"OUTPUT_TOUR_FILE = {out_path}\n")
        f.write(f"RUNS = {lkh_runs}\n")
        f.write(f"SEED = {seed}\n")
        # Optional knobs (uncomment if you want):
        # f.write("TRACE_LEVEL = 0\n")  # less logging
        # f.write("MOVE_TYPE = 5\n")
        # f.write("PATCHING_C = 3\n")
        # f.write("PATCHING_A = 2\n")

def read_tour_file(tour_path):
    """
    Returns tour as 0-based list of node indices (length n)
    """
    tour = []
    in_section = False
    with open(tour_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line == "TOUR_SECTION":
                in_section = True
                continue
            if not in_section:
                continue
            if line == "-1" or line == "EOF":
                break
            tour.append(int(line) - 1)  # to 0-based
    return tour

def solve_with_lkh(coords_float, LKH_PATH, scale=1000000, runs=1, seed=1234):
    """
    coords_float: (n,2) float array in [0,1]
    returns: (tour_0_based, tour_file_path)
    """
    coords_int = np.rint(coords_float * scale).astype(np.int64)

    with tempfile.TemporaryDirectory() as td:
        tsp_path  = os.path.join(td, "inst.tsp")
        par_path  = os.path.join(td, "inst.par")
        tour_path = os.path.join(td, "inst.tour")

        write_tsplib_euc2d(tsp_path, coords_int, name="inst")
        write_lkh_par(par_path, tsp_path, tour_path, lkh_runs=runs, seed=seed)

        # Run LKH
        # LKH prints to stdout; we keep it quiet here
        subprocess.run([LKH_PATH, par_path], check=True, stdout=subprocess.DEVNULL)

        tour = read_tour_file(tour_path)
        return tour

def runLKH(val_dataset):
    LKH_PATH = "../LKH-3.0.13/LKH"
    costs = []
    routes = []
    for coords in val_dataset:
        coords_np = coords.cpu().numpy()
        tour = solve_with_lkh(coords_np, LKH_PATH, runs=1)
        cost = np.sum(
            np.linalg.norm(
                coords_np[tour] - coords_np[np.roll(tour, -1)],
                axis=1
            )
        )
        costs.append(cost)
        routes.append(tour)
    avg_cost = sum(costs)/len(costs)
    print(f"LKH average cost over {len(val_dataset)} instances: {avg_cost}")
    return costs, routes

Learn heuristic for TSP100: 

In [14]:

n_node = 100
n_ants = 100
k_sparse = 32
H=20

net = Net().to(device)
net.load_state_dict(torch.load(f'../pretrained/tsp/tsp{n_node}.pt', map_location=device))
val_dataset = load_test_dataset(n_node=n_node, device=device)
# validation(n_ants, net, val_dataset, k_sparse, True)

costs, routes = runLKH(val_dataset)

7.797036
7.837153
7.846018
8.106737
8.045571
7.238635
7.656117
7.727924
7.342738
7.5919986
7.789987
7.5224957
7.7608166
7.663014
7.655603
7.491823
7.9816527
8.198282
7.873479
7.7760196
8.068875
7.5542364
7.9876285
7.8805146
8.272972
7.4053164
7.8655996
7.73015
7.9356318
7.2303867
7.458157
7.630615
7.9517956
7.797214
7.6675334
8.112659
8.04118
7.9811854
7.9608507
7.860474
7.7310305
7.565384
7.984146
7.8702474
8.086998
7.924321
7.831262
7.871598
7.766511
7.9028664
7.5188465
7.7549424
7.581902
7.419349
8.192114
7.311065
8.059754
7.695738
7.679855
7.726414
7.521006
7.6358376
7.692862
8.166603
7.7087493
7.635141
7.88309
7.9172406
7.861652
7.3978415
7.357586
7.8662167
7.9962916
7.6564918
7.968583
7.491341
8.275138
7.925787
7.726292
7.3740306
7.996146
7.8140907
7.945643
7.871953
7.812144
7.3762193
7.7335324
8.152571
8.1349
7.535629
7.6994514
7.667531
7.4127207
7.692391
7.6123924
8.033801
7.8495603
7.5412755
8.1587515
7.3729706
7.860874
7.4816775
7.7064867
7.532342
7.6978116
7.8590174
7.924301

Learn heuristic for TSP500: 

In [ ]:

n_node = 200
n_ants = 100
k_sparse = 32
steps_per_epoch = 64
epochs = 10

net = Net().to(device)
net.load_state_dict(torch.load(f'../pretrained/tsp/tsp{n_node}.pt', map_location=device))
val_dataset = load_val_dataset(n_node=n_node, device=device)
validation(n_ants, net, val_dataset, k_sparse, True)

(11.014584422111511, 10.747489601373672)

In [ ]:

n_node = 500
n_ants = 100
k_sparse = 32
steps_per_epoch = 64
epochs = 10

net = Net().to(device)
net.load_state_dict(torch.load(f'../pretrained/tsp/tsp{n_node}.pt', map_location=device))
val_dataset = load_val_dataset(n_node=n_node, device=device)
validation(n_ants, net, val_dataset, k_sparse, True)

(16.83272649347782, 16.672277003526688)

In [ ]:

n_node = 1000
n_ants = 100
k_sparse = 32
steps_per_epoch = 64
epochs = 10

net = Net().to(device)
net.load_state_dict(torch.load(f'../pretrained/tsp/tsp{n_node}.pt', map_location=device))
val_dataset = load_val_dataset(n_node=n_node, device=device)
validation(n_ants, net, val_dataset, k_sparse, True)

(23.547117590904236, 23.395665407180786)

In [ ]:

n_node = 2000
n_ants = 100
k_sparse = 32
steps_per_epoch = 64
epochs = 10

net = Net().to(device)
net.load_state_dict(torch.load(f'../pretrained/tsp/tsp{n_node}.pt', map_location=device))
val_dataset = load_val_dataset(n_node=n_node, device=device)
validation(n_ants, net, val_dataset, k_sparse, True)

(33.000582218170166, 32.91691970825195)

In [ ]:

n_node = 5000
n_ants = 100
k_sparse = 32
steps_per_epoch = 64
epochs = 10

net = Net().to(device)
net.load_state_dict(torch.load(f'../pretrained/tsp/tsp{n_node}.pt', map_location=device))
val_dataset = load_val_dataset(n_node=n_node, device=device)
validation(n_ants, net, val_dataset, k_sparse, True)

(52.50774621963501, 52.44724941253662)